Clean the Arabic text and remove any remaining personal information, so the posts are ready for feature extraction.

## Step 1: Load the cleaned posts
Load the output of notebook 01 (patient, date, text, label).

In [1]:
import re
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "phase1_data" else Path.cwd()
df = pd.read_csv(ROOT / "data" / "interim" / "clean_basic.csv", dtype={"text": str}, encoding="utf-8-sig")
df["date"] = pd.to_datetime(df["date"])

funnel = [("Loaded", len(df), df["patient"].nunique())]
print("Posts:", len(df), "| Patients:", df["patient"].nunique())
print("Columns:", list(df.columns))

Posts: 48542 | Patients: 209
Columns: ['patient', 'post_time', 'text', 'date', 'group', 'moca', 'label']


## Step 2: Check the text before cleaning
Measure how much of the text contains emojis, links, phone numbers, emails and mentions, so we know what needs cleaning.

In [2]:
checks = {
    "links":    r"https?://\S+|www\.\S+",
    "emails":   r"\S+@\S+\.\S+",
    "mentions": r"@\w+",
    "phones":   r"\+?\d[\d\s\-]{7,}\d",
    "long numbers (6+ digits)": r"\d{6,}",
    "emojis":   "[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF]",
    "English letters": r"[A-Za-z]",
    "Arabic letters":  r"[\u0600-\u06FF]",
}
for name, pat in checks.items():
    n = df["text"].str.contains(pat, regex=True, na=False).sum()
    print(f"{name:26s} {n:6d} posts ({n / len(df) * 100:.1f}%)")

links                         885 posts (1.8%)
emails                         19 posts (0.0%)
mentions                       46 posts (0.1%)
phones                        707 posts (1.5%)
long numbers (6+ digits)      738 posts (1.5%)
emojis                      10755 posts (22.2%)
English letters             11052 posts (22.8%)
Arabic letters              47178 posts (97.2%)


97% Arabic, so Arabic normalization is essential
22% have emojis: we’ll count them (a possible feature), then remove them
1.8% have links, 1.5% phone-like numbers, 1.5% long numbers: these must be removed/masked for privacy
Emails and mentions are rare, but we’ll still mask them
23% have English letters: partly links, partly the patient IDs (P001…) that replaced names. We’ll recheck after removing links

## Step 3: Count and remove emojis
22% of posts contain emojis. We count them per post first (emoji use may be a writing-style feature), then remove them from the text.

In [3]:
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF\U0001F1E6-\U0001F1FF\u200d\ufe0f]+")

df["emoji_count"] = df["text"].str.count(EMOJI_RE.pattern)
df["text"] = df["text"].str.replace(EMOJI_RE, " ", regex=True)

print("Posts with emojis:", (df["emoji_count"] > 0).sum())
print("Emojis left in text:", df["text"].str.contains(EMOJI_RE.pattern, regex=True, na=False).sum())

Posts with emojis: 10803
Emojis left in text: 0


## Step 4: Remove links and mask personal information
Remove links, and replace emails, @mentions, phone numbers and long ID-like numbers with tags ([EMAIL], [PERSON], [PHONE], [ID]), so no personal details remain while the sentence shape is kept.

In [4]:
before = {
    "links":    df["text"].str.contains(r"https?://\S+|www\.\S+", regex=True, na=False).sum(),
    "emails":   df["text"].str.contains(r"\S+@\S+\.\S+", regex=True, na=False).sum(),
    "mentions": df["text"].str.contains(r"@\w+", regex=True, na=False).sum(),
    "phones":   df["text"].str.contains(r"\+?\d[\d\s\-]{7,}\d", regex=True, na=False).sum(),
}

df["text"] = (df["text"]
    .str.replace(r"https?://\S+|www\.\S+", " ", regex=True)
    .str.replace(r"\S+@\S+\.\S+", " [EMAIL] ", regex=True)
    .str.replace(r"@\w+", " [PERSON] ", regex=True)
    .str.replace(r"\+?\d[\d\s\-]{7,}\d", " [PHONE] ", regex=True)
    .str.replace(r"\d{6,}", " [ID] ", regex=True))

after = {
    "links":    df["text"].str.contains(r"https?://\S+|www\.\S+", regex=True, na=False).sum(),
    "emails":   df["text"].str.contains(r"\S+@\S+\.\S+", regex=True, na=False).sum(),
    "mentions": df["text"].str.contains(r"@\w+", regex=True, na=False).sum(),
    "phones":   df["text"].str.contains(r"\+?\d[\d\s\-]{7,}\d", regex=True, na=False).sum(),
}
for k in before:
    print(f"{k:9s} before: {before[k]:5d} | after: {after[k]}")

links     before:   885 | after: 0
emails    before:    19 | after: 0
mentions  before:    46 | after: 0
phones    before:   712 | after: 0


## Step 5: Count and shorten elongated letters
Stretched words like "ههههه" or "كثيييير" are counted first (they may reflect writing style), then shortened to two letters so the same word is always written the same way.

In [5]:
ELONG_RE = re.compile(r"(.)\1{2,}")

df["elongation_count"] = df["text"].str.count(ELONG_RE.pattern)
df["text"] = df["text"].str.replace(ELONG_RE, r"\1\1", regex=True)

print("Posts with elongated letters:", (df["elongation_count"] > 0).sum())
print("Elongations left:", df["text"].str.contains(ELONG_RE.pattern, regex=True, na=False).sum())

Posts with elongated letters: 14110


C:\Users\talah\AppData\Local\Temp\ipykernel_31860\3176769445.py:7: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  print("Elongations left:", df["text"].str.contains(ELONG_RE.pattern, regex=True, na=False).sum())


Elongations left: 0


## Step 6: Arabic normalization
Unify letter forms (أ إ آ ٱ → ا, ى → ي, ة → ه, ؤ → و, ئ → ي), and remove diacritics (tashkeel) and stretching (tatweel), so the same word is always written the same way.

In [6]:
TASHKEEL = r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]"
ARABIC_NORMALIZE = {"أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا", "ى": "ي", "ة": "ه", "ؤ": "و", "ئ": "ي"}

before_tashkeel = df["text"].str.contains(TASHKEEL, regex=True, na=False).sum()

df["text"] = df["text"].str.replace(TASHKEEL, "", regex=True).str.replace("\u0640", "", regex=False)
for a, b in ARABIC_NORMALIZE.items():
    df["text"] = df["text"].str.replace(a, b, regex=False)

print("Posts that had diacritics:", before_tashkeel)
print("Diacritics left:", df["text"].str.contains(TASHKEEL, regex=True, na=False).sum())
print("Letter variants left:", df["text"].str.contains("[أإآٱىةؤئ]", regex=True, na=False).sum())

Posts that had diacritics: 8213
Diacritics left: 0
Letter variants left: 0


## Step 7: Clean punctuation, spaces and patient IDs
Unify Arabic and English punctuation, remove extra spaces, and replace patient IDs that appear inside the text (where names used to be) with [SELF], so the model can't use the ID as a clue.

In [7]:
ids_in_text = df["text"].str.contains(r"\bP\d{3}\b", regex=True, na=False).sum()

df["text"] = (df["text"]
    .str.replace(r"\bP\d{3}\b", " [SELF] ", regex=True)
    .str.replace("،", ",", regex=False)
    .str.replace("؛", ";", regex=False)
    .str.replace("؟", "?", regex=False)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip())

print("Posts that contained a patient ID:", ids_in_text)
print("IDs left in text:", df["text"].str.contains(r"\bP\d{3}\b", regex=True, na=False).sum())
print("Posts with English letters now:", df["text"].str.contains(r"[A-Za-z]", regex=True, na=False).sum())

Posts that contained a patient ID: 338
IDs left in text: 0
Posts with English letters now: 10788


## Step 8: Check how much real English text remains
Count posts with English words (excluding our privacy tags), to decide whether English needs separate handling.

In [8]:
no_tags = df["text"].str.replace(r"\[(EMAIL|PERSON|PHONE|ID|SELF)\]", " ", regex=True)
has_en = no_tags.str.contains(r"[A-Za-z]{2,}", regex=True, na=False)
mostly_en = no_tags.str.count(r"[A-Za-z]") > no_tags.str.count(r"[\u0600-\u06FF]")

print("Posts with some English words:", has_en.sum(), f"({has_en.mean() * 100:.1f}%)")
print("Posts mostly in English:", mostly_en.sum(), f"({mostly_en.mean() * 100:.1f}%)")

Posts with some English words: 10177 (21.0%)
Posts mostly in English: 1559 (3.2%)


## Step 9: Lowercase English and flag English posts
Only 3.2% of posts are mostly English, so Arabic stays the main language. English letters are lowercased so the same word is counted once, and mostly-English posts are flagged.

In [9]:
df["mostly_english"] = mostly_en.astype(int)
df["text"] = df["text"].str.replace(r"[A-Za-z]+", lambda m: m.group(0).lower(), regex=True)
# keep our tags in their original form
df["text"] = df["text"].str.replace(r"\[(email|person|phone|id|self)\]",
                                    lambda m: m.group(0).upper(), regex=True)

print("Mostly-English posts flagged:", df["mostly_english"].sum())
print("Uppercase letters left (excluding tags):",
      df["text"].str.replace(r"\[(EMAIL|PERSON|PHONE|ID|SELF)\]", "", regex=True)
                .str.contains(r"[A-Z]", regex=True, na=False).sum())

Mostly-English posts flagged: 1559
Uppercase letters left (excluding tags): 0


## Step 10: Remove posts that mention the diagnosis
Posts mentioning Alzheimer's or dementia would reveal the label directly, so the model would "cheat" by reading the word instead of learning language changes. These posts are removed.

In [10]:
# Whole words only (so e.g. "خرفان" = sheep is NOT matched)
DIAG_RE = r"(?<!\w)(?:ال)?(?:زهايمر|خرف)(?!\w)|alzheimer\w*|dementia"

has_diag = df["text"].str.contains(DIAG_RE, regex=True, na=False)
print("Posts mentioning the diagnosis:", has_diag.sum())
print("By group:")
print(df.loc[has_diag, "group"].value_counts())

before = len(df)
df = df[~has_diag]
funnel.append(("No diagnosis mentions", len(df), df["patient"].nunique()))
print("Removed:", before - len(df), "| now:", len(df), "| patients:", df["patient"].nunique())


Posts mentioning the diagnosis: 24
By group:
group
MCI         15
Normal       8
Dementia     1
Name: count, dtype: int64
Removed: 24 | now: 48518 | patients: 209


**Result:** 24 posts mentioning Alzheimer's or dementia removed (15 MCI, 8 Normal, 1 Dementia); 48,518 posts remain.

## Step 11: Remove posts with no real words
After cleaning, some posts may contain only tags, numbers or punctuation. Posts with no real words are removed, since there is no language to analyze.

In [12]:
no_tags = df["text"].str.replace(r"\[(EMAIL|PERSON|PHONE|ID|SELF)\]", " ", regex=True)
df["n_words"] = no_tags.str.count(r"[\u0600-\u06FFa-z]{2,}")

print("Posts with 0 real words:", (df["n_words"] == 0).sum())
print("Posts with 1–2 words:", df["n_words"].between(1, 2).sum())

before = len(df)
df = df[df["n_words"] > 0]
funnel.append(("Posts with real words", len(df), df["patient"].nunique()))
print("Removed:", before - len(df), "| now:", len(df), "| patients:", df["patient"].nunique())

Posts with 0 real words: 583
Posts with 1–2 words: 2589
Removed: 583 | now: 47935 | patients: 209


## Step 12: Privacy check by eye
Read a random sample of cleaned posts to confirm no names, phone numbers or other personal details remain. 

In [13]:
sample = df.sample(40, random_state=7)[["patient", "text"]]
for _, r in sample.iterrows():
    print(r["patient"], "|", r["text"][:200])
    print("-" * 60)

P137 | *معلومات ابهرتني* هل تعلم ما هي الخماسيات .. من دقه القران الكريم تلك الخماسيات التي جعلها الله في كتابه في افتتاحيات السور لتسهيل معرفه بدايات السور في القران الكريم وتشمل الفيات السبع التاليه : الفي
------------------------------------------------------------
P138 | اعز الولد ولد الولد جدي مفيش حاجه في الدنيا تعوضه
------------------------------------------------------------
P144 | بفوز بيراميدز عل الاهلي امس ..اثبت انه يستحق المركز الثاني بعد الزمالك
------------------------------------------------------------
P017 | الجزء الثاني من مناقشه كتب عبدالرحمن منصور الاستاذ فتحي نجم fathy negm
------------------------------------------------------------
P017 | العباسه
------------------------------------------------------------
P075 | الروعه وجمال سويسرا
------------------------------------------------------------
P062 | امين يارب العالمين
------------------------------------------------------------
P048 | لله تعالي *فضل الدعاء للمسلم بظهر الغيب* *النبي صلي الله عليه وسلم علمنا الدع

## Step 13: Mask names after titles and remove Facebook markers
Names that follow titles (الاستاذ، الشيخ، د، دكتور، الحاج، المهندس، ابو، ام) are replaced with [PERSON], and Facebook's "… more" cut-off marker is removed.

In [14]:
TITLES = r"(?:الاستاذ|استاذ|الاستاذه|الشيخ|شيخ|الدكتور|دكتور|الدكتوره|د\s*/|د\.|الحاج|الحاجه|المهندس|م\s*/|ابو|ام)"
NAME_AFTER_TITLE = rf"(?<!\w){TITLES}\s+[\u0600-\u06FF]+(?:\s+[\u0600-\u06FF]+)?"

before_titles = df["text"].str.contains(NAME_AFTER_TITLE, regex=True, na=False).sum()
df["text"] = df["text"].str.replace(NAME_AFTER_TITLE, " [PERSON] ", regex=True)

df["text"] = (df["text"]
    .str.replace(r"…\s*more|\.\.\.\s*more|\bmore\b$", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True).str.strip())

print("Posts with a name after a title (masked):", before_titles)
print("'more' markers left:", df["text"].str.contains(r"…\s*more", regex=True, na=False).sum())

Posts with a name after a title (masked): 3648
'more' markers left: 0


In [15]:
TITLES = r"(?:الاستاذ|استاذ|الاستاذه|الشيخ|شيخ|الدكتور|دكتور|الدكتوره|د\s*/|د\.|الحاج|الحاجه|المهندس|م\s*/)"

In [16]:
TITLES = r"(?:الاستاذ|استاذ|الاستاذه|الشيخ|شيخ|الدكتور|دكتور|الدكتوره|د\s*/|د\.|الحاج|الحاجه|المهندس|م\s*/)"
NAME_AFTER_TITLE = rf"(?<!\w){TITLES}\s+[\u0600-\u06FF]+(?:\s+[\u0600-\u06FF]+)?"

before_titles = df["text"].str.contains(NAME_AFTER_TITLE, regex=True, na=False).sum()
df["text"] = df["text"].str.replace(NAME_AFTER_TITLE, " [PERSON] ", regex=True)

df["text"] = (df["text"]
    .str.replace(r"…\s*more|\.\.\.\s*more|\bmore\b$", " ", regex=True)
    .str.replace(r"\s+", " ", regex=True).str.strip())

print("Posts with a name after a title (masked):", before_titles)
print("'more' markers left:", df["text"].str.contains(r"…\s*more", regex=True, na=False).sum())

Posts with a name after a title (masked): 15
'more' markers left: 0


## Step 14: Flag copied content
Many posts are copied texts (religious quotes, sayings, shared articles), not the patient's own words. These are flagged so the language features can be computed on the patient's own writing.

In [17]:
COPIED_RE = (
    r"رواه|صلي الله عليه وسلم|قال تعالي|قال الله|سبحانه وتعالي|"   # religious citations
    r"منقول|#|"                                                     # "copied" marker, hashtags
    r"\*[^*]{20,}\*|\"[^\"]{40,}\"|«[^»]{40,}»"                     # long text inside * * or quotes
)

df["copied"] = (
    df["text"].str.contains(COPIED_RE, regex=True, na=False)
    | (df["n_words"] > 120)                                          # very long posts are usually copied
).astype(int)

print("Posts flagged as copied:", df["copied"].sum(), f"({df['copied'].mean() * 100:.1f}%)")
print("Own-words posts:", (df["copied"] == 0).sum())
print()
print("Copied share by group (%):")
print((df.groupby("group")["copied"].mean() * 100).round(1))
own = df[df["copied"] == 0].groupby("patient").size()
print()
print("Patients with fewer than 5 own-words posts:", (own < 5).sum() + (209 - own.size))

Posts flagged as copied: 7887 (16.5%)
Own-words posts: 40048

Copied share by group (%):
group
Dementia    13.4
MCI         16.6
Normal      17.4
Name: copied, dtype: float64

Patients with fewer than 5 own-words posts: 0


**Result:** 7,887 posts (16.5%) flagged as copied content, similar across groups (13–17%). 40,048 own-words posts remain, and every patient keeps at least 5.

## Step 15: Cleaning funnel
Summary of how many posts and patients remained after each NLP cleaning step.

In [18]:
funnel.append(("Own-words posts (copied flagged)", int((df["copied"] == 0).sum()), df["patient"].nunique()))
funnel_df = pd.DataFrame(funnel, columns=["step", "rows", "patients"])
funnel_df

,step,rows,patients
0,Loaded,48542,209
1,No diagnosis mentions,48518,209
2,Posts with real words,47935,209
3,Own-words posts (copied flagged),40048,209


## Step 16: Save the clean posts
Save the cleaned, anonymized posts for feature extraction (notebook 03). Saved in `data/interim`, never pushed to GitHub.

In [19]:
cols = ["patient", "date", "text", "n_words", "emoji_count", "elongation_count",
        "mostly_english", "copied", "group", "moca", "label"]
out = ROOT / "data" / "interim" / "clean_posts.csv"
df[cols].to_csv(out, index=False, encoding="utf-8-sig")
funnel_df.to_csv(ROOT / "phase1_data" / "nlp_funnel.csv", index=False)
print("Saved:", out.relative_to(ROOT), "| posts:", len(df), "| patients:", df["patient"].nunique())

Saved: data\interim\clean_posts.csv | posts: 47935 | patients: 209
